# Purpose: Creating coordinate files for promoters and genes
Taking RefSeq coordinates for genes and parsing out: 
1. Promoter coordinates 
2. Gene coordinates for
each unique TSS and TES (alternative splicing isoforms are ignored) 
3. One set of coordinates for each gene body, using the most internal TSS and TES, and removing the first and last 1 kb so as to remove sites of
polymerase pausing

In [77]:
import pandas as pd
import numpy as np
import seaborn as sns


#amount upstream and downstream of annotated TSS to consider a promoter
lowerBound=1000
upperBound=1000

chromList = ['chrX', 'chrY']
chromList = chromList + [f'chr{i}' for i in range(1,23)]


In [78]:
# input
refseqFile = '../../Manuscript_data/hg38_refseq.txt'
outputs = '../../figure_outputs/'

# outputs

#BED file that has one of coordinate set per gene, 
#using most internal TSS and TES, removing 1 kb from 5' and 3' ends
gbFile=outputs+'hg38_refseq_NR_gene_bodies.bed' 

promoterFile=outputs+'hg38_refseq_promoters.bed'

In [79]:
rs_df = pd.read_csv(refseqFile, sep='\t')
rs_df

,#bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,score,name2,cdsStartStat,cdsEndStat,exonFrames
0,0,XM_011541469.1,chr1,-,67092175,67109072,67093004,67103382,5,"67092175,67095234,67096251,67103237,67109028,","67093604,67095421,67096321,67103382,67109072,",0,C1orf141,cmpl,cmpl,"0,2,1,0,-1,"
1,0,XM_011541467.1,chr1,-,67092175,67131183,67093004,67127240,9,"67092175,67095234,67096251,67103237,67111576,6...","67093604,67095421,67096321,67103343,67111644,6...",0,C1orf141,cmpl,cmpl,"0,2,1,0,1,2,0,0,-1,"
2,0,XM_017001276.1,chr1,-,67092175,67131227,67093004,67127240,9,"67092175,67095234,67096251,67103237,67111576,6...","67093604,67095421,67096321,67103382,67111644,6...",0,C1orf141,cmpl,cmpl,"0,2,1,0,1,2,0,0,-1,"
3,0,XM_011541465.2,chr1,-,67092175,67134962,67093004,67127240,9,"67092175,67095234,67096251,67103237,67111576,6...","67093604,67095421,67096321,67103382,67111644,6...",0,C1orf141,cmpl,cmpl,"0,2,1,0,1,2,0,0,-1,"
4,0,NR_075077.1,chr1,-,67092175,67134971,67134971,67134971,10,"67092175,67096251,67103237,67111576,67113613,6...","67093604,67096321,67103382,67111644,67113756,6...",0,C1orf141,none,none,"-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167464,586,XM_017030167.1,chr22_KI270734v1_random,-,138085,153840,138479,152899,13,"138085,138742,142193,143613,144748,145003,1466...","138667,138831,142292,143789,144895,145096,1467...",0,LOC102724788,cmpl,cmpl,"1,2,2,0,0,0,2,0,0,0,1,1,0,"
167465,586,XR_951398.2,chr22_KI270734v1_random,-,138085,161588,161588,161588,14,"138085,138742,142193,143613,144748,145003,1466...","138667,138831,142292,143789,144929,145096,1467...",0,LOC102724788,none,none,"-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,"
167466,586,XM_017030168.1,chr22_KI270734v1_random,-,138085,161592,138479,156446,14,"138085,138742,142193,143613,144748,145003,1466...","138667,138831,142292,143789,144895,145096,1467...",0,LOC102724788,cmpl,cmpl,"1,2,2,0,0,0,2,0,0,1,1,2,0,-1,"
167467,586,XM_006724936.3,chr22_KI270734v1_random,-,138085,161594,138479,150995,13,"138085,138742,142193,143613,144748,145003,1466...","138667,138831,142292,143789,144895,145096,1467...",0,LOC102724788,cmpl,cmpl,"1,2,2,0,0,0,2,0,0,0,0,-1,-1,"


In [80]:
# keep NM transcripts in typical chromosomes
nm_mask = rs_df['name'].str.split('_', expand=True)[0] == 'NM'
rs_df = rs_df.loc[nm_mask]
rs_df = rs_df.loc[rs_df['chrom'].isin(chromList)].reset_index(drop=True)
rs_df

,#bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,score,name2,cdsStartStat,cdsEndStat,exonFrames
0,0,NM_001276352.1,chr1,-,67092175,67134971,67093579,67127240,9,"67092175,67096251,67103237,67111576,67115351,6...","67093604,67096321,67103382,67111644,67115464,6...",0,C1orf141,cmpl,cmpl,"2,1,0,1,2,0,0,-1,-1,"
1,0,NM_001276351.1,chr1,-,67092175,67134971,67093004,67127240,8,"67092175,67095234,67096251,67115351,67125751,6...","67093604,67095421,67096321,67115464,67125909,6...",0,C1orf141,cmpl,cmpl,"0,2,1,2,0,0,-1,-1,"
2,0,NM_001005337.2,chr1,+,201283451,201332993,201283702,201328836,14,"201283451,201293941,201313165,201316552,201317...","201283904,201294045,201313560,201316697,201317...",0,PKP1,cmpl,cmpl,"0,1,0,2,0,1,2,0,0,0,1,2,0,-1,"
3,0,NM_000299.3,chr1,+,201283451,201332993,201283702,201328836,15,"201283451,201293941,201313165,201316552,201317...","201283904,201294045,201313560,201316697,201317...",0,PKP1,cmpl,cmpl,"0,1,0,2,0,1,2,2,0,0,0,1,2,0,-1,"
4,1,NM_001042682.1,chr1,-,8352403,8423687,8355086,8364133,13,"8352403,8355418,8356099,8358195,8359763,836011...","8355120,8355599,8356246,8358916,8359986,836149...",0,RERE,cmpl,cmpl,"2,1,1,0,2,0,0,0,0,-1,-1,-1,-1,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50005,972,NM_001130922.2,chr22,-,50767491,50783682,50768775,50782294,9,"50767491,50769040,50769454,50769904,50775771,5...","50768874,50769124,50769549,50770016,50775851,5...",0,RABL2B,cmpl,cmpl,"0,0,1,0,1,2,2,0,-1,"
50006,972,NM_001350010.1,chr22,-,50767491,50783682,50768775,50782294,9,"50767491,50769040,50769424,50769904,50775771,5...","50768874,50769124,50769552,50770016,50775851,5...",0,RABL2B,cmpl,cmpl,"0,0,1,0,1,2,2,0,-1,"
50007,972,NM_001130919.2,chr22,-,50767491,50783682,50768775,50782294,9,"50767491,50769040,50769454,50769904,50775771,5...","50768874,50769124,50769552,50770016,50775851,5...",0,RABL2B,cmpl,cmpl,"0,0,1,0,1,2,2,0,-1,"
50008,972,NM_001350015.1,chr22,-,50767491,50783682,50768775,50782294,9,"50767491,50769040,50769424,50769904,50775771,5...","50768874,50769124,50769549,50770016,50775851,5...",0,RABL2B,cmpl,cmpl,"0,0,1,0,1,2,2,0,-1,"


In [81]:
rs_bed = rs_df[['chrom','txStart','txEnd','name2']]
rs_bed['score'] = '.'
rs_bed['strand'] = rs_df['strand']
rs_bed

/var/folders/x2/34lg9m394nj1zx1ph2bkj0tw0000gn/T/ipykernel_37222/546778032.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rs_bed['score'] = '.'
/var/folders/x2/34lg9m394nj1zx1ph2bkj0tw0000gn/T/ipykernel_37222/546778032.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rs_bed['strand'] = rs_df['strand']


,chrom,txStart,txEnd,name2,score,strand
0,chr1,67092175,67134971,C1orf141,.,-
1,chr1,67092175,67134971,C1orf141,.,-
2,chr1,201283451,201332993,PKP1,.,+
3,chr1,201283451,201332993,PKP1,.,+
4,chr1,8352403,8423687,RERE,.,-
...,...,...,...,...,...,...
50005,chr22,50767491,50783682,RABL2B,.,-
50006,chr22,50767491,50783682,RABL2B,.,-
50007,chr22,50767491,50783682,RABL2B,.,-
50008,chr22,50767491,50783682,RABL2B,.,-


In [82]:
genes = list(rs_bed['name2'].unique())
rs_bed_gb = pd.DataFrame(np.zeros((len(genes),6)))
promoters = []


In [114]:
def gb_coords(df, lowerBound, upperBound, gene):
    chr = df['chrom'].unique()
    start = df['txStart'].max() + lowerBound
    end = df['txEnd'].min() - upperBound
    strand = df['strand'].unique()
    if len(chr) > 1:
        print(df)
    
    return pd.Series((chr, start, end,gene, '.', strand))

In [115]:
def prom_coords(df, lowerBound, upperBound, gene):
    chr = df['chrom'].unique()[0]
    strand = df['strand'].unique()[0]
    if strand == '+':
        starts = df['txStart'].unique()
    else:
        starts = df['txEnd'].unique()
    toReturn = []
    for start in starts:
        toReturn.append([chr, start - lowerBound, start + upperBound, gene, '.', strand])
    return toReturn

In [116]:
for i, gene in enumerate(genes):
    gene_mask = rs_bed['name2'] == gene

    rs_bed_gb.iloc[i,:] = gb_coords(
        rs_bed.loc[gene_mask],
        lowerBound,
        upperBound,
        gene
    )
    
    promoters = promoters + prom_coords(
        rs_bed.loc[gene_mask],
        lowerBound,
        upperBound,
        gene
    )



    

      chrom  txStart   txEnd name2 score strand
24321  chrX   624343  659411  SHOX     .      +
24850  chrX   624343  646823  SHOX     .      +
26110  chrY   624343  659411  SHOX     .      +
26161  chrY   624343  646823  SHOX     .      +
      chrom  txStart    txEnd  name2 score strand
24322  chrX  1403138  1452977  ASMTL     .      -
24323  chrX  1403138  1452977  ASMTL     .      -
24324  chrX  1403138  1453762  ASMTL     .      -
26111  chrY  1403138  1452977  ASMTL     .      -
26112  chrY  1403138  1452977  ASMTL     .      -
26113  chrY  1403138  1453762  ASMTL     .      -
      chrom  txStart    txEnd  name2 score strand
24325  chrX  2219505  2500974  DHRSX     .      -
26114  chrY  2219505  2500974  DHRSX     .      -
      chrom  txStart    txEnd  name2 score strand
24326  chrX  2486413  2500539  ZBED1     .      -
24327  chrX  2486413  2500967  ZBED1     .      -
24328  chrX  2486413  2500967  ZBED1     .      -
26115  chrY  2486413  2500539  ZBED1     .      -
26116  chr

In [86]:
pd.DataFrame(promoters)

,0,1,2,3,4,5
0,chr1,67133971,67135971,C1orf141,.,-
1,chr1,201282451,201284451,PKP1,.,+
2,chr1,8422687,8424687,RERE,.,-
3,chr1,8816640,8818640,RERE,.,-
4,chr1,34164274,34166274,CSMD2,.,-
...,...,...,...,...,...,...
26503,chr22,50581999,50583999,CHKB,.,-
26504,chr22,50599684,50601684,MAPK8IP2,.,+
26505,chr22,50627173,50629173,ARSA,.,-
26506,chr22,50737223,50739223,ACR,.,+


In [87]:
rs_bed_gb

,0,1,2,3,4,5
0,chr1,67093175.0,67133971.0,C1orf141,.,-
1,chr1,201284451.0,201331993.0,PKP1,.,+
2,chr1,8353403.0,8422687.0,RERE,.,-
3,chr1,33514998.0,34164274.0,CSMD2,.,-
4,chr1,75207389.0,75610114.0,SLC44A5,.,-
...,...,...,...,...,...,...
19422,chr22,50579957.0,50581999.0,CHKB,.,-
19423,chr22,50601684.0,50612981.0,MAPK8IP2,.,+
19424,chr22,50623753.0,50627173.0,ARSA,.,-
19425,chr22,50739223.0,50744299.0,ACR,.,+


In [88]:
rs_bed_gb.to_csv(gbFile, header=None, index=None)
pd.DataFrame(promoters).to_csv(promoterFile, header=None, index=None)

In [89]:
orig_path = '../../Manuscript_data/'
orig_prom = f"{orig_path}hg38_refseq_promoters.bed"
orig_gb = f"{orig_path}hg38_refseq_NR_1kb.bed"

In [90]:
orig_prom_df = pd.read_table(orig_prom, header=None)
orig_prom_df

,0,1,2,3,4,5
0,chr1,67133971,67135971,C1orf141,.,-
1,chr1,201282451,201284451,PKP1,.,+
2,chr1,8422687,8424687,RERE,.,-
3,chr1,8816640,8818640,RERE,.,-
4,chr1,34164274,34166274,CSMD2,.,-
...,...,...,...,...,...,...
26503,chr22,50581999,50583999,CHKB,.,-
26504,chr22,50599684,50601684,MAPK8IP2,.,+
26505,chr22,50627173,50629173,ARSA,.,-
26506,chr22,50737223,50739223,ACR,.,+


In [91]:
promoters_df = pd.DataFrame(promoters)
promoters_df

,0,1,2,3,4,5
0,chr1,67133971,67135971,C1orf141,.,-
1,chr1,201282451,201284451,PKP1,.,+
2,chr1,8422687,8424687,RERE,.,-
3,chr1,8816640,8818640,RERE,.,-
4,chr1,34164274,34166274,CSMD2,.,-
...,...,...,...,...,...,...
26503,chr22,50581999,50583999,CHKB,.,-
26504,chr22,50599684,50601684,MAPK8IP2,.,+
26505,chr22,50627173,50629173,ARSA,.,-
26506,chr22,50737223,50739223,ACR,.,+


In [92]:
promoters_set = set(promoters_df[0]+':'+promoters_df[1].astype(str)+'-'+promoters_df[2].astype(str))
orig_prom_set = set(orig_prom_df[0]+':'+orig_prom_df[1].astype(str)+'-'+orig_prom_df[2].astype(str))

In [93]:
promoters_set - orig_prom_set

{'chrX:56953254-56955254', 'chrX:57066799-57068799', 'chrX:57183100-57185100'}

In [94]:
orig_prom_set - promoters_set

{'chrY:56953254-56955254', 'chrY:57066799-57068799', 'chrY:57183100-57185100'}

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [95]:
rs_bed_gb[1] = rs_bed_gb[1].astype(int)
rs_bed_gb[2] = rs_bed_gb[2].astype(int)

orig_gb_df = pd.read_table(orig_gb, header=None)
orig_gb_df[2] = orig_gb_df[2] - 2000

gb_set = set(rs_bed_gb[0]+':'+rs_bed_gb[1].astype(str)+'-'+rs_bed_gb[2].astype(str))
orig_gb_set = set(orig_gb_df[0]+':'+orig_gb_df[1].astype(str)+'-'+orig_gb_df[2].astype(str))

TypeError: unhashable type: 'numpy.ndarray'

In [72]:
orig_gb_df.sort_values(by=3)

,0,1,2,3,4,5
3909,chr19,58347805,58352499,A1BG,.,-
15742,chr10,50800408,50884675,A1CF,.,-
8172,chr12,9068707,9115229,A2M,.,-
8824,chr12,8846003,8875783,A2ML1,.,+
1835,chr1,33307765,33320098,A3GALT2,.,-
...,...,...,...,...,...,...
15970,chr1,52843510,52893575,ZYG11A,.,+
9235,chr1,52727458,52826342,ZYG11B,.,+
5649,chr7,143382266,143390111,ZYX,.,+
19205,chr17,4005444,4141959,ZZEF1,.,-


In [73]:
rs_bed_gb.sort_values(by=3)

,0,1,2,3,4,5
18173,chr19,58347805,58352499,A1BG,.,-
10187,chr10,50800408,50884675,A1CF,.,-
12477,chr12,9068707,9115229,A2M,.,-
12473,chr12,8846003,8875783,A2ML1,.,+
941,chr1,33307765,33320098,A3GALT2,.,-
...,...,...,...,...,...,...
1108,chr1,52843510,52893575,ZYG11A,.,+
244,chr1,52727458,52826342,ZYG11B,.,+
7714,chr7,143382266,143390111,ZYX,.,+
15427,chr17,4005444,4141959,ZZEF1,.,-


In [74]:
rs_bed.loc[rs_bed['name2'] == 'A1BG']

,chrom,txStart,txEnd,name2,score,strand
46797,chr19,58346805,58353499,A1BG,.,-


In [75]:
orig_gb_set - gb_set

{'chrX:155768734-56967979',
 'chrX:155882279-57129289',
 'chrX:155998580-57196337',
 'chrY:12702230-12859843',
 'chrY:12906704-12919478',
 'chrY:13324033-13479670',
 'chrY:13704566-13705024',
 'chrY:13986771-13985512',
 'chrY:14057221-14055958',
 'chrY:14623020-14732549',
 'chrY:17769979-17769560',
 'chrY:17879259-17879219',
 'chrY:18026786-18026746',
 'chrY:18136448-18136029',
 'chrY:18547690-18547592',
 'chrY:18772814-18772715',
 'chrY:19706414-19743939',
 'chrY:20576710-20592154',
 'chrY:20757067-20780032',
 'chrY:21383960-21385360',
 'chrY:21512337-21524554',
 'chrY:21535878-21548326',
 'chrY:21881075-21893526',
 'chrY:21904617-21917027',
 'chrY:22072755-22095007',
 'chrY:22169541-22181942',
 'chrY:22404409-22416649',
 'chrY:22491396-22513637',
 'chrY:22985262-23004465',
 'chrY:23130354-23198092',
 'chrY:23220456-23290356',
 'chrY:24046792-24047014',
 'chrY:24619003-24638207',
 'chrY:24764068-24812492',
 'chrY:24834819-24906040',
 'chrY:25031900-25051104',
 'chrY:25623116-25623338'

In [76]:
gb_set - orig_gb_set

{'chrX:155768734-155781459',
 'chrX:155882279-155942769',
 'chrX:155998580-156009817'}

In [118]:
orig_gb_df.loc[orig_gb_df[3] == 'SHOX']

,0,1,2,3,4,5
6106,chrX,625343,645823,SHOX,.,+


In [120]:
rs_df.loc[rs_df['name2'] == 'SHOX']

,#bin,name,chrom,strand,txStart,txEnd,cdsStart,cdsEnd,exonCount,exonStarts,exonEnds,score,name2,cdsStartStat,cdsEndStat,exonFrames
24321,73,NM_006883.2,chrX,+,624343,659411,630897,658829,6,"624343,630465,634617,640820,640998,658784,","624602,631174,634826,640878,641087,659411,",0,SHOX,cmpl,cmpl,"-1,0,1,0,1,0,"
24850,589,NM_000451.3,chrX,+,624343,646823,630897,644636,6,"624343,630465,634617,640820,640998,644390,","624602,631174,634826,640878,641087,646823,",0,SHOX,cmpl,cmpl,"-1,0,1,0,1,0,"
26110,73,NM_006883.2,chrY,+,624343,659411,630897,658829,6,"624343,630465,634617,640820,640998,658784,","624602,631174,634826,640878,641087,659411,",0,SHOX,cmpl,cmpl,"-1,0,1,0,1,0,"
26161,589,NM_000451.3,chrY,+,624343,646823,630897,644636,6,"624343,630465,634617,640820,640998,644390,","624602,631174,634826,640878,641087,646823,",0,SHOX,cmpl,cmpl,"-1,0,1,0,1,0,"
